# Interactive Atom Grid With AOD Lines

Choose the grid size and atom type, then click a grid intersection to place an atom. The visualization is drawn in diagonal/anti-diagonal coordinates: `f_h` lines are one family of diagonals, `f_v` lines are the other family. Each iteration can activate selected `f_h` and `f_v` lines with integer frequency shifts, apply Hadamard highlights to selected atoms, and/or run a Rydberg highlight over nearby atoms. Fixed green ovals can be added between node pairs and are drawn underneath atoms.

Use the visual scaling sliders under the canvas to adjust the drawn window size, atom size, and oval width/length without changing grid coordinates.

The rule controls sit to the right of the canvas while there is enough horizontal space; if the canvas is enlarged or the output area is narrow, the controls wrap below it.

The rotated grid uses equal horizontal and vertical projection scale, so the cells remain true squares after the 45-degree rotation.

Use Add iteration to extend the program beyond the initial five steps; saved JSON stores the full dynamic program.


In [ ]:

from IPython.display import HTML, display
import uuid


widget_id = f"atom-grid-{uuid.uuid4().hex[:8]}"

html = """
<div id='__ID__' style='font-family: sans-serif; width: 100%; max-width: none; --grid-window-size: 1300px;'>
  <style>
    #__ID__ button { margin: 2px; }
    #__ID__ input, #__ID__ select { margin-top: 3px; }
    #__ID__ .panel { border: 1px solid #ddd; padding: 10px; margin-top: 10px; }
    #__ID__ .row { display: flex; gap: 10px; align-items: flex-end; flex-wrap: wrap; }
    #__ID__ label { font-size: 13px; }
    #__ID__ .main-layout { display: flex; flex-wrap: wrap; gap: 18px; align-items: flex-start; width: 100%; }
    #__ID__ .grid-pane { flex: 0 0 var(--grid-window-size); max-width: none; }
    #__ID__ .controls-pane { flex: 1 1 430px; min-width: 360px; max-width: 760px; margin-top: 0; }
  </style>
  <div class='main-layout'>
    <div class='grid-pane'>
      <div class='row' style='margin-bottom: 10px;'>
        <div>
          <label for='__ID__-size'><b>Grid size N</b></label><br>
          <input id='__ID__-size' type='number' min='2' max='30' value='8' style='width: 74px;'>
        </div>
        <div>
          <label for='__ID__-type'><b>Atom type</b></label><br>
          <select id='__ID__-type'>
            <option value='data'>data</option>
            <option value='ancilla'>ancilla</option>
          </select>
        </div>
        <div>
          <button id='__ID__-remove'>Remove selected node</button>
          <button id='__ID__-clear'>Clear grid</button>
        </div>
      </div>
      <div class='row' style='margin-bottom: 10px;'>
        <button id='__ID__-step'>Step selected iteration</button>
        <button id='__ID__-run'>Run all iterations</button>
        <button id='__ID__-repeat'>Repeat all iterations</button>
        <button id='__ID__-undo-step'>Undo last step</button>
        <button id='__ID__-undo-all'>Undo all steps</button>
        <button id='__ID__-save'>Save JSON</button>
        <label style='display: inline-block;'>
          <span style='display:none;'>Load JSON</span>
          <input id='__ID__-load' type='file' accept='.json,application/json' style='width: 185px;'>
        </label>
      </div>
      <div id='__ID__-status' style='margin-bottom: 10px;'>Selected node: none</div>
      <canvas id='__ID__-canvas' width='1300' height='1300' style='border: 1px solid #ddd; background: white; cursor: crosshair; width: 1300px; height: 1300px; max-width: none;'></canvas>
      <div style='margin-top: 10px; font-size: 14px;'>
        Coordinates are shown as <code>(f_h, f_v)</code>. A movement step affects an atom only when both its current <code>f_h</code> line and current <code>f_v</code> line are active in the selected iteration.
      </div>
      <div class='panel' style='max-width: var(--grid-window-size);'>
        <div style='margin-bottom: 8px;'><b>Visual scaling</b></div>
        <div style='display: grid; grid-template-columns: 130px 1fr 52px; gap: 8px; align-items: center;'>
          <label for='__ID__-window-size'>Window</label>
          <input id='__ID__-window-size' type='range' min='500' max='1500' step='25' value='1300'>
          <span id='__ID__-window-size-value'>1300</span>
          <label for='__ID__-atom-scale'>Atoms</label>
          <input id='__ID__-atom-scale' type='range' min='40' max='220' step='5' value='100'>
          <span id='__ID__-atom-scale-value'>100%</span>
          <label for='__ID__-oval-width-scale'>Oval width</label>
          <input id='__ID__-oval-width-scale' type='range' min='50' max='300' step='5' value='100'>
          <span id='__ID__-oval-width-scale-value'>100%</span>
          <label for='__ID__-oval-length-scale'>Oval length</label>
          <input id='__ID__-oval-length-scale' type='range' min='50' max='300' step='5' value='100'>
          <span id='__ID__-oval-length-scale-value'>100%</span>
        </div>
      </div>
    </div>
    <div class='controls-pane'>
      <div class='panel' style='margin-top: 0;'>
        <div class='row'>
          <div>
            <label for='__ID__-iteration'><b>Active iteration</b></label><br>
            <select id='__ID__-iteration'>
              <option value='0'>Iteration 1</option>
              <option value='1'>Iteration 2</option>
              <option value='2'>Iteration 3</option>
              <option value='3'>Iteration 4</option>
              <option value='4'>Iteration 5</option>
            </select>
          </div>
          <div><button id='__ID__-add-iteration'>Add iteration</button></div>
          <div id='__ID__-iteration-summary' style='font-size: 14px; color: #333;'>Iteration 1</div>
        </div>
      </div>

      <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 12px;'>
        <div class='panel'>
          <label for='__ID__-atoms'><b>Atoms / qubits</b></label><br>
          <select id='__ID__-atoms' multiple size='12' style='width: 100%;'></select>
        </div>
        <div class='panel'>
          <label for='__ID__-lines'><b>Active AOD lines</b></label><br>
          <select id='__ID__-lines' size='12' style='width: 100%;'></select>
          <div style='margin-top: 8px;'>
            <button id='__ID__-remove-line'>Remove line</button>
            <button id='__ID__-clear-lines'>Clear lines</button>
          </div>
        </div>
      </div>

      <div class='panel'>
        <div style='margin-bottom: 8px;'><b>Add/update movement line</b></div>
        <div class='row'>
          <div>
            <label for='__ID__-axis'>Axis</label><br>
            <select id='__ID__-axis'>
              <option value='fh'>f_h</option>
              <option value='fv'>f_v</option>
            </select>
          </div>
          <div>
            <label for='__ID__-line-index'>Line index</label><br>
            <input id='__ID__-line-index' type='number' value='0' step='1' min='0' style='width: 78px;'>
          </div>
          <div>
            <label for='__ID__-line-delta'>delta</label><br>
            <input id='__ID__-line-delta' type='number' value='0' step='1' style='width: 78px;'>
          </div>
          <div><button id='__ID__-set-line'>Set line</button></div>
        </div>
      </div>

      <div class='panel'>
        <div style='margin-bottom: 8px;'><b>Hadamard / Rydberg</b></div>
        <div class='row'>
          <button id='__ID__-set-h'>Set Hadamard on selected atoms</button>
          <button id='__ID__-clear-h'>Clear Hadamard</button>
          <label><input id='__ID__-rydberg' type='checkbox'> Rydberg in this iteration</label>
        </div>
        <div id='__ID__-h-summary' style='font-size: 13px; margin-top: 6px; color: #333;'></div>
      </div>

      <div class='panel'>
        <div style='margin-bottom: 8px;'><b>Fixed ovals</b></div>
        <div class='row'>
          <div><label for='__ID__-oval-fh1'>x1/f_h1</label><br><input id='__ID__-oval-fh1' type='number' value='0' min='0' step='1' style='width: 72px;'></div>
          <div><label for='__ID__-oval-fv1'>y1/f_v1</label><br><input id='__ID__-oval-fv1' type='number' value='0' min='0' step='1' style='width: 72px;'></div>
          <div><label for='__ID__-oval-fh2'>x2/f_h2</label><br><input id='__ID__-oval-fh2' type='number' value='1' min='0' step='1' style='width: 72px;'></div>
          <div><label for='__ID__-oval-fv2'>y2/f_v2</label><br><input id='__ID__-oval-fv2' type='number' value='1' min='0' step='1' style='width: 72px;'></div>
          <div><button id='__ID__-add-oval'>Add oval</button></div>
        </div>
        <select id='__ID__-ovals' size='5' style='width: 100%; margin-top: 8px;'></select>
        <div style='margin-top: 8px;'><button id='__ID__-delete-oval'>Delete oval</button></div>
      </div>
    </div>
  </div>
</div>

<script>
(function() {
  const INITIAL_ITERATION_COUNT = 5;
  const root = document.getElementById('__ID__');
  const canvas = document.getElementById('__ID__-canvas');
  const ctx = canvas.getContext('2d');
  const sizeInput = document.getElementById('__ID__-size');
  const typeInput = document.getElementById('__ID__-type');
  const removeButton = document.getElementById('__ID__-remove');
  const clearButton = document.getElementById('__ID__-clear');
  const stepButton = document.getElementById('__ID__-step');
  const runButton = document.getElementById('__ID__-run');
  const repeatButton = document.getElementById('__ID__-repeat');
  const undoStepButton = document.getElementById('__ID__-undo-step');
  const undoAllButton = document.getElementById('__ID__-undo-all');
  const saveButton = document.getElementById('__ID__-save');
  const loadInput = document.getElementById('__ID__-load');
  const status = document.getElementById('__ID__-status');
  const iterationSelect = document.getElementById('__ID__-iteration');
  const iterationSummary = document.getElementById('__ID__-iteration-summary');
  const addIterationButton = document.getElementById('__ID__-add-iteration');
  const atomsSelect = document.getElementById('__ID__-atoms');
  const linesSelect = document.getElementById('__ID__-lines');
  const axisInput = document.getElementById('__ID__-axis');
  const lineIndexInput = document.getElementById('__ID__-line-index');
  const lineDeltaInput = document.getElementById('__ID__-line-delta');
  const setLineButton = document.getElementById('__ID__-set-line');
  const removeLineButton = document.getElementById('__ID__-remove-line');
  const clearLinesButton = document.getElementById('__ID__-clear-lines');
  const setHButton = document.getElementById('__ID__-set-h');
  const clearHButton = document.getElementById('__ID__-clear-h');
  const rydbergInput = document.getElementById('__ID__-rydberg');
  const hSummary = document.getElementById('__ID__-h-summary');
  const ovalFh1Input = document.getElementById('__ID__-oval-fh1');
  const ovalFv1Input = document.getElementById('__ID__-oval-fv1');
  const ovalFh2Input = document.getElementById('__ID__-oval-fh2');
  const ovalFv2Input = document.getElementById('__ID__-oval-fv2');
  const addOvalButton = document.getElementById('__ID__-add-oval');
  const ovalsSelect = document.getElementById('__ID__-ovals');
  const deleteOvalButton = document.getElementById('__ID__-delete-oval');
  const windowSizeInput = document.getElementById('__ID__-window-size');
  const atomScaleInput = document.getElementById('__ID__-atom-scale');
  const ovalWidthScaleInput = document.getElementById('__ID__-oval-width-scale');
  const ovalLengthScaleInput = document.getElementById('__ID__-oval-length-scale');
  const windowSizeValue = document.getElementById('__ID__-window-size-value');
  const atomScaleValue = document.getElementById('__ID__-atom-scale-value');
  const ovalWidthScaleValue = document.getElementById('__ID__-oval-width-scale-value');
  const ovalLengthScaleValue = document.getElementById('__ID__-oval-length-scale-value');

  const colors = { data: '#d62728', ancilla: '#1f77ff' };
  let gridSize = Number(sizeInput.value);
  let selectedNode = null;
  let atoms = [];
  let nextAtomId = { data: 1, ancilla: 1 };
  let nextAtomUid = 1;
  let ovals = [];
  let nextOvalId = 1;
  let program = defaultProgram();
  let animationState = null;
  let activeEffect = null;
  let lastProgramConfigured = false;
  let stepHistory = [];
  let visualSettings = { windowSize: 1300, atomScale: 1, ovalWidthScale: 1, ovalLengthScale: 1 };
  let requestedIterationIndex = null;

  function cloneAtoms(atomList = atoms) {
    return atomList.map(atom => ({ ...atom }));
  }

  function clearUndoHistory() {
    stepHistory = [];
  }

  function emptyIteration() {
    return { fh: {}, fv: {}, hadamardUids: [], rydberg: false };
  }

  function defaultProgram() {
    return Array.from({ length: INITIAL_ITERATION_COUNT }, () => emptyIteration());
  }

  function iterationCount() {
    return program.length;
  }

  function activeIterationIndex() {
    return Math.max(0, Math.min(iterationCount() - 1, Number(iterationSelect.value) || 0));
  }

  function activeIterationLabel(index = activeIterationIndex()) {
    return `Iteration ${index + 1}`;
  }

  function key(fh, fv) {
    return `${fh},${fv}`;
  }

  function gridCenter() {
    return { x: canvas.width / 2, y: canvas.height / 2 };
  }

  function gridStep() {
    return gridSize > 1 ? Math.min((canvas.width - 90) / (2 * (gridSize - 1)), (canvas.height - 90) / (2 * (gridSize - 1))) : 0;
  }

  function clamp(value, min, max) {
    return Math.max(min, Math.min(max, value));
  }

  function atomRadius() {
    return clamp(gridStep() * 0.23 * visualSettings.atomScale, 2, 24);
  }

  function atomFontSize() {
    return clamp(atomRadius() * 0.9, 5, 16);
  }

  function hadamardRadius() {
    return atomRadius() * 1.8;
  }

  function rydbergRadius() {
    return atomRadius() * 2.0;
  }

  function selectedAtomRadius() {
    return atomRadius() * 1.5;
  }

  function pairOffset() {
    return atomRadius() * 1.15;
  }

  function ovalPadding() {
    return clamp(gridStep() * 0.30 * visualSettings.ovalLengthScale, 3, 80);
  }

  function ovalMinorRadius() {
    return clamp(gridStep() * 0.34 * visualSettings.ovalWidthScale, 3, 80);
  }

  function nodeToPixel(fh, fv) {
    const center = gridCenter();
    const step = gridStep();
    return {
      x: center.x + (fv - fh) * step,
      y: center.y + (fh + fv - (gridSize - 1)) * step,
    };
  }

  function selectedLabel() {
    return selectedNode ? `(${selectedNode.fh}, ${selectedNode.fv})` : 'none';
  }

  function sortAtoms(list = atoms) {
    return list.slice().sort((a, b) => a.type.localeCompare(b.type) || a.id - b.id || a.uid - b.uid);
  }

  function atomsAtNode(fh, fv, list = atoms) {
    return list.filter(atom => atom.fh === fh && atom.fv === fv);
  }

  function atomCounts() {
    return {
      data: atoms.filter(atom => atom.type === 'data').length,
      ancilla: atoms.filter(atom => atom.type === 'ancilla').length,
    };
  }

  function renumberAtoms() {
    ['data', 'ancilla'].forEach(atomType => {
      let nextId = 1;
      atoms
        .filter(atom => atom.type === atomType)
        .sort((a, b) => a.id - b.id || a.uid - b.uid)
        .forEach(atom => {
          atom.id = nextId;
          nextId += 1;
        });
      nextAtomId[atomType] = nextId;
    });
  }

  function getIteration(index = activeIterationIndex()) {
    while (!program[index]) {
      program.push(emptyIteration());
    }
    program[index].fh = program[index].fh || {};
    program[index].fv = program[index].fv || {};
    program[index].hadamardUids = program[index].hadamardUids || [];
    program[index].rydberg = Boolean(program[index].rydberg);
    return program[index];
  }

  function selectedAtomUids() {
    return Array.from(atomsSelect.selectedOptions).map(option => Number(option.value));
  }

  function selectedLine() {
    if (!linesSelect.value) {
      return null;
    }
    const [axis, index] = linesSelect.value.split(':');
    return { axis, index };
  }

  function selectedOvalId() {
    return ovalsSelect.value ? Number(ovalsSelect.value) : null;
  }

  function atomLabel(atom) {
    return `${atom.type} #${atom.id} uid=${atom.uid} @ (${atom.fh}, ${atom.fv})`;
  }

  function refreshIterationOptions() {
    const selectedIndex = requestedIterationIndex === null
      ? activeIterationIndex()
      : Math.max(0, Math.min(iterationCount() - 1, requestedIterationIndex));
    requestedIterationIndex = null;
    iterationSelect.innerHTML = '';
    program.forEach((_, index) => {
      const option = document.createElement('option');
      option.value = String(index);
      option.textContent = `Iteration ${index + 1}`;
      option.selected = index === selectedIndex;
      iterationSelect.appendChild(option);
    });
  }

  function refreshAtomsList() {
    const selected = new Set(selectedAtomUids());
    atomsSelect.innerHTML = '';
    sortAtoms().forEach(atom => {
      const option = document.createElement('option');
      option.value = String(atom.uid);
      option.textContent = atomLabel(atom);
      option.selected = selected.has(atom.uid);
      atomsSelect.appendChild(option);
    });
  }

  function refreshLinesList() {
    const selected = selectedLine();
    const selectedValue = selected ? `${selected.axis}:${selected.index}` : null;
    const step = getIteration();
    const rows = [];
    Object.keys(step.fh).sort((a, b) => Number(a) - Number(b)).forEach(index => {
      rows.push({ value: `fh:${index}`, text: `f_h_${index}: delta f_h=${step.fh[index]}` });
    });
    Object.keys(step.fv).sort((a, b) => Number(a) - Number(b)).forEach(index => {
      rows.push({ value: `fv:${index}`, text: `f_v_${index}: delta f_v=${step.fv[index]}` });
    });
    linesSelect.innerHTML = '';
    rows.forEach(row => {
      const option = document.createElement('option');
      option.value = row.value;
      option.textContent = row.text;
      option.selected = row.value === selectedValue;
      linesSelect.appendChild(option);
    });
  }

  function refreshOvalsList() {
    const selected = selectedOvalId();
    ovalsSelect.innerHTML = '';
    ovals.forEach(oval => {
      const option = document.createElement('option');
      option.value = String(oval.id);
      option.textContent = `#${oval.id}: (${oval.fh1}, ${oval.fv1}) - (${oval.fh2}, ${oval.fv2})`;
      option.selected = selected === oval.id;
      ovalsSelect.appendChild(option);
    });
  }

  function refreshIterationControls() {
    const step = getIteration();
    rydbergInput.checked = Boolean(step.rydberg);
    const hAtoms = step.hadamardUids
      .map(uid => atoms.find(atom => atom.uid === uid))
      .filter(Boolean)
      .map(atom => `${atom.type} #${atom.id}`);
    hSummary.textContent = `Hadamard: ${hAtoms.length ? hAtoms.join(', ') : 'none'}`;
    const fhCount = Object.keys(step.fh).length;
    const fvCount = Object.keys(step.fv).length;
    iterationSummary.textContent = `${activeIterationLabel()} / ${iterationCount()}: f_h lines ${fhCount}, f_v lines ${fvCount}, Hadamard ${hAtoms.length}, Rydberg ${step.rydberg ? 'on' : 'off'}`;
  }

  function updateStatus(message = '') {
    const counts = atomCounts();
    status.innerHTML = `Selected node: <b>${selectedLabel()}</b> | active: <b>${activeIterationLabel()}</b> | data: <b>${counts.data}</b>, ancilla: <b>${counts.ancilla}</b>, ovals: <b>${ovals.length}</b>, undo: <b>${stepHistory.length}</b>${message ? ' | ' + message : ''}`;
  }

  function getNodeOccupancy(atomList = atoms) {
    const occupancy = new Map();
    atomList.forEach(atom => {
      const nodeKey = key(atom.fh, atom.fv);
      if (!occupancy.has(nodeKey)) {
        occupancy.set(nodeKey, []);
      }
      occupancy.get(nodeKey).push(atom);
    });
    occupancy.forEach(nodeAtoms => {
      nodeAtoms.sort((a, b) => a.type.localeCompare(b.type) || a.id - b.id || a.uid - b.uid);
    });
    return occupancy;
  }

  function offsetForIndex(index, count) {
    if (count <= 1) {
      return { x: 0, y: 0 };
    }
    const offset = pairOffset();
    return index === 0 ? { x: -offset, y: 0 } : { x: offset, y: 0 };
  }

  function atomPixel(atom, occupancy = getNodeOccupancy()) {
    const nodeKey = key(atom.fh, atom.fv);
    const nodeAtoms = occupancy.get(nodeKey) || [atom];
    const index = Math.max(0, nodeAtoms.findIndex(candidate => candidate.uid === atom.uid));
    const offset = offsetForIndex(index, nodeAtoms.length);
    const base = nodeToPixel(atom.fh, atom.fv);
    return { x: base.x + offset.x, y: base.y + offset.y };
  }

  function lineEndpoints(axis, index) {
    if (axis === 'fh') {
      return [nodeToPixel(index, 0), nodeToPixel(index, gridSize - 1)];
    }
    return [nodeToPixel(0, index), nodeToPixel(gridSize - 1, index)];
  }

  function drawLine(axis, index, style = {}) {
    const [a, b] = lineEndpoints(axis, index);
    ctx.save();
    ctx.strokeStyle = style.color || 'black';
    ctx.lineWidth = style.width || 1;
    if (style.dash) {
      ctx.setLineDash(style.dash);
    }
    ctx.beginPath();
    ctx.moveTo(a.x, a.y);
    ctx.lineTo(b.x, b.y);
    ctx.stroke();
    ctx.restore();
  }

  function drawGrid() {
    for (let i = 0; i < gridSize; i += 1) {
      drawLine('fh', i, { color: '#111', width: 1 });
      drawLine('fv', i, { color: '#111', width: 1 });
    }
    ctx.save();
    ctx.fillStyle = '#222';
    ctx.font = '12px sans-serif';
    ctx.textAlign = 'center';
    ctx.textBaseline = 'middle';
    for (let i = 0; i < gridSize; i += 1) {
      const fhLabel = nodeToPixel(i, 0);
      const fvLabel = nodeToPixel(0, i);
      ctx.fillText(`f_h${i}`, fhLabel.x - 24, fhLabel.y - 8);
      ctx.fillText(`f_v${i}`, fvLabel.x + 24, fvLabel.y - 8);
    }
    ctx.restore();
  }

  function drawSelectedNode() {
    if (!selectedNode) {
      return;
    }
    const pixel = nodeToPixel(selectedNode.fh, selectedNode.fv);
    ctx.beginPath();
    ctx.arc(pixel.x, pixel.y, selectedAtomRadius(), 0, 2 * Math.PI);
    ctx.strokeStyle = '#666';
    ctx.lineWidth = 2;
    ctx.stroke();
  }

  function drawActiveLines(step, progress = 0) {
    Object.keys(step.fh).forEach(index => {
      const movedIndex = Number(index) + Number(step.fh[index]) * progress;
      drawLine('fh', movedIndex, { color: '#0067ff', width: 2, dash: [8, 7] });
    });
    Object.keys(step.fv).forEach(index => {
      const movedIndex = Number(index) + Number(step.fv[index]) * progress;
      drawLine('fv', movedIndex, { color: '#0067ff', width: 2, dash: [8, 7] });
    });
  }

  function ovalGeometry(oval) {
    const p1 = nodeToPixel(oval.fh1, oval.fv1);
    const p2 = nodeToPixel(oval.fh2, oval.fv2);
    const cx = (p1.x + p2.x) / 2;
    const cy = (p1.y + p2.y) / 2;
    const dx = p2.x - p1.x;
    const dy = p2.y - p1.y;
    const distance = Math.hypot(dx, dy);
    return {
      cx,
      cy,
      angle: Math.atan2(dy, dx),
      rx: Math.max(ovalMinorRadius(), distance / 2 + ovalPadding()),
      ry: ovalMinorRadius(),
    };
  }

  function atomInsideOval(atom, oval) {
    const occupancy = getNodeOccupancy();
    const p = atomPixel(atom, occupancy);
    const g = ovalGeometry(oval);
    const cos = Math.cos(-g.angle);
    const sin = Math.sin(-g.angle);
    const dx = p.x - g.cx;
    const dy = p.y - g.cy;
    const x = dx * cos - dy * sin;
    const y = dx * sin + dy * cos;
    return (x * x) / (g.rx * g.rx) + (y * y) / (g.ry * g.ry) <= 1;
  }

  function activeRydbergAtomUids() {
    const uids = new Set();
    for (let i = 0; i < atoms.length; i += 1) {
      for (let j = i + 1; j < atoms.length; j += 1) {
        const a = atoms[i];
        const b = atoms[j];
        const close = Math.max(Math.abs(a.fh - b.fh), Math.abs(a.fv - b.fv)) <= 1;
        if (close) {
          uids.add(a.uid);
          uids.add(b.uid);
        }
      }
    }
    return uids;
  }

  function activeRydbergOvalIds() {
    const ids = new Set();
    ovals.forEach(oval => {
      const inside = atoms.filter(atom => atomInsideOval(atom, oval));
      if (inside.length >= 2) {
        ids.add(oval.id);
      }
    });
    return ids;
  }

  function drawOvals() {
    const blackOvalIds = activeEffect && activeEffect.rydberg ? activeEffect.ovalIds : new Set();
    ovals.forEach(oval => {
      const g = ovalGeometry(oval);
      ctx.save();
      ctx.translate(g.cx, g.cy);
      ctx.rotate(g.angle);
      ctx.beginPath();
      ctx.ellipse(0, 0, g.rx, g.ry, 0, 0, 2 * Math.PI);
      if (blackOvalIds.has(oval.id)) {
        ctx.fillStyle = 'rgba(0, 0, 0, 0.34)';
        ctx.strokeStyle = 'rgba(0, 0, 0, 0.85)';
      } else {
        ctx.fillStyle = 'rgba(0, 160, 80, 0.20)';
        ctx.strokeStyle = 'rgba(0, 130, 70, 0.55)';
      }
      ctx.fill();
      ctx.lineWidth = 2;
      ctx.stroke();
      ctx.restore();
    });
  }

  function drawAtoms(atomList = atoms, occupancy = getNodeOccupancy(atomList), framePositions = null) {
    const highlighted = new Set(selectedAtomUids());
    const hadamardUids = activeEffect ? activeEffect.hadamardUids : new Set();
    const rydbergUids = activeEffect && activeEffect.rydberg ? activeEffect.rydbergUids : new Set();
    sortAtoms(atomList).forEach(atom => {
      const pixel = framePositions && framePositions.has(atom.uid) ? framePositions.get(atom.uid) : atomPixel(atom, occupancy);
      if (hadamardUids.has(atom.uid)) {
        ctx.beginPath();
        ctx.arc(pixel.x, pixel.y, hadamardRadius(), 0, 2 * Math.PI);
        ctx.fillStyle = 'rgba(255, 215, 0, 0.58)';
        ctx.fill();
        ctx.strokeStyle = 'rgba(170, 130, 0, 0.9)';
        ctx.lineWidth = 2;
        ctx.stroke();
      }
      if (rydbergUids.has(atom.uid)) {
        ctx.beginPath();
        ctx.arc(pixel.x, pixel.y, rydbergRadius(), 0, 2 * Math.PI);
        ctx.fillStyle = 'rgba(0, 185, 90, 0.42)';
        ctx.fill();
        ctx.strokeStyle = 'rgba(0, 120, 55, 0.95)';
        ctx.lineWidth = 2;
        ctx.stroke();
      }
      if (highlighted.has(atom.uid)) {
        ctx.beginPath();
        ctx.arc(pixel.x, pixel.y, selectedAtomRadius(), 0, 2 * Math.PI);
        ctx.strokeStyle = '#333';
        ctx.lineWidth = 2;
        ctx.stroke();
      }
      ctx.beginPath();
      ctx.arc(pixel.x, pixel.y, atomRadius(), 0, 2 * Math.PI);
      ctx.fillStyle = colors[atom.type];
      ctx.fill();
      ctx.strokeStyle = 'black';
      ctx.lineWidth = 1;
      ctx.stroke();
      ctx.fillStyle = 'white';
      ctx.font = `bold ${atomFontSize()}px sans-serif`;
      ctx.textAlign = 'center';
      ctx.textBaseline = 'middle';
      ctx.fillText(String(atom.id), pixel.x, pixel.y + 0.5);
    });
  }

  function applyWindowSize() {
    const size = Math.floor(Number(windowSizeInput.value) || 700);
    visualSettings.windowSize = size;
    canvas.width = size;
    canvas.height = size;
    canvas.style.width = `${size}px`;
    canvas.style.height = `${size}px`;
    root.style.setProperty('--grid-window-size', `${size}px`);
    windowSizeValue.textContent = String(size);
  }

  function updateVisualSettings() {
    visualSettings.atomScale = (Number(atomScaleInput.value) || 100) / 100;
    visualSettings.ovalWidthScale = (Number(ovalWidthScaleInput.value) || 100) / 100;
    visualSettings.ovalLengthScale = (Number(ovalLengthScaleInput.value) || 100) / 100;
    atomScaleValue.textContent = `${Math.round(visualSettings.atomScale * 100)}%`;
    ovalWidthScaleValue.textContent = `${Math.round(visualSettings.ovalWidthScale * 100)}%`;
    ovalLengthScaleValue.textContent = `${Math.round(visualSettings.ovalLengthScale * 100)}%`;
    applyWindowSize();
  }

  function handleVisualControlChange() {
    updateVisualSettings();
    drawScene();
  }

  function drawScene() {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    drawGrid();
    drawSelectedNode();
    drawOvals();

    if (animationState && animationState.phase === 'running') {
      drawActiveLines(animationState.step, animationState.progress);
      drawAtoms(animationState.sourceAtoms, animationState.startOccupancy, animationState.framePositions);
      return;
    }

    drawActiveLines(getIteration(), 0);
    drawAtoms();
  }

  function refreshUi(message = '') {
    refreshIterationOptions();
    refreshAtomsList();
    refreshLinesList();
    refreshOvalsList();
    refreshIterationControls();
    updateStatus(message);
    drawScene();
  }

  function nearestNode(event) {
    const rect = canvas.getBoundingClientRect();
    const scaleX = canvas.width / rect.width;
    const scaleY = canvas.height / rect.height;
    const x = (event.clientX - rect.left) * scaleX;
    const y = (event.clientY - rect.top) * scaleY;
    let best = { fh: 0, fv: 0, d: Infinity };
    for (let fh = 0; fh < gridSize; fh += 1) {
      for (let fv = 0; fv < gridSize; fv += 1) {
        const p = nodeToPixel(fh, fv);
        const d = Math.hypot(x - p.x, y - p.y);
        if (d < best.d) {
          best = { fh, fv, d };
        }
      }
    }
    return { fh: best.fh, fv: best.fv };
  }

  function addAtomAt(fh, fv) {
    if (animationState) {
      updateStatus('wait until animation finishes');
      return;
    }
    if (atomsAtNode(fh, fv).length >= 2) {
      refreshUi('node already has two atoms');
      return;
    }
    const type = typeInput.value;
    const atom = { uid: nextAtomUid, type, id: nextAtomId[type], fh, fv };
    nextAtomUid += 1;
    nextAtomId[type] += 1;
    atoms.push(atom);
    clearUndoHistory();
    lastProgramConfigured = false;
    refreshUi(`added ${type} #${atom.id} at (${fh}, ${fv})`);
  }

  function removeSelectedNode() {
    if (animationState) {
      updateStatus('wait until animation finishes');
      return;
    }
    if (!selectedNode) {
      refreshUi('select a node first');
      return;
    }
    const nodeAtoms = atomsAtNode(selectedNode.fh, selectedNode.fv).sort((a, b) => b.uid - a.uid);
    if (nodeAtoms.length === 0) {
      refreshUi('selected node is already empty');
      return;
    }
    const removed = nodeAtoms[0];
    atoms = atoms.filter(atom => atom.uid !== removed.uid);
    program.forEach(step => {
      step.hadamardUids = step.hadamardUids.filter(uid => uid !== removed.uid);
    });
    renumberAtoms();
    clearUndoHistory();
    lastProgramConfigured = false;
    refreshUi(`removed ${removed.type} #${removed.id}`);
  }

  function clearGrid() {
    if (animationState) {
      updateStatus('wait until animation finishes');
      return;
    }
    atoms = [];
    nextAtomId = { data: 1, ancilla: 1 };
    nextAtomUid = 1;
    selectedNode = null;
    program.forEach(step => { step.hadamardUids = []; });
    clearUndoHistory();
    lastProgramConfigured = false;
    refreshUi('cleared grid');
  }

  function resizeGrid() {
    if (animationState) {
      updateStatus('wait until animation finishes');
      sizeInput.value = gridSize;
      return;
    }
    const newSize = Math.max(2, Math.min(30, Number(sizeInput.value) || 8));
    sizeInput.value = newSize;
    gridSize = newSize;
    atoms = atoms.filter(atom => atom.fh < gridSize && atom.fv < gridSize);
    ovals = ovals.filter(oval => oval.fh1 < gridSize && oval.fv1 < gridSize && oval.fh2 < gridSize && oval.fv2 < gridSize);
    program.forEach(step => {
      Object.keys(step.fh).forEach(index => { if (Number(index) >= gridSize) delete step.fh[index]; });
      Object.keys(step.fv).forEach(index => { if (Number(index) >= gridSize) delete step.fv[index]; });
      step.hadamardUids = step.hadamardUids.filter(uid => atoms.some(atom => atom.uid === uid));
    });
    renumberAtoms();
    clearUndoHistory();
    if (selectedNode && (selectedNode.fh >= gridSize || selectedNode.fv >= gridSize)) {
      selectedNode = null;
    }
    lastProgramConfigured = false;
    refreshUi(`resized grid to ${gridSize} x ${gridSize}`);
  }

  function setMovementLine() {
    const step = getIteration();
    const axis = axisInput.value;
    const index = Math.floor(Number(lineIndexInput.value));
    const delta = Math.floor(Number(lineDeltaInput.value) || 0);
    if (index < 0 || index >= gridSize) {
      refreshUi('line index is outside the grid');
      return;
    }
    step[axis][String(index)] = delta;
    lastProgramConfigured = false;
    refreshUi(`set ${axis === 'fh' ? 'f_h' : 'f_v'}_${index} delta ${delta}`);
  }

  function removeMovementLine() {
    const selected = selectedLine();
    if (!selected) {
      refreshUi('select a line first');
      return;
    }
    const step = getIteration();
    delete step[selected.axis][selected.index];
    lastProgramConfigured = false;
    refreshUi('removed active line');
  }

  function clearMovementLines() {
    const step = getIteration();
    step.fh = {};
    step.fv = {};
    lastProgramConfigured = false;
    refreshUi('cleared movement lines');
  }

  function setHadamard() {
    const selected = selectedAtomUids();
    const step = getIteration();
    step.hadamardUids = selected;
    lastProgramConfigured = false;
    refreshUi(`Hadamard set for ${selected.length} atom(s)`);
  }

  function clearHadamard() {
    const step = getIteration();
    step.hadamardUids = [];
    lastProgramConfigured = false;
    refreshUi('Hadamard cleared');
  }

  function setRydbergFromInput() {
    const step = getIteration();
    step.rydberg = rydbergInput.checked;
    lastProgramConfigured = false;
    refreshUi(`Rydberg ${step.rydberg ? 'enabled' : 'disabled'}`);
  }

  function validNode(fh, fv) {
    return Number.isInteger(fh) && Number.isInteger(fv) && fh >= 0 && fh < gridSize && fv >= 0 && fv < gridSize;
  }

  function addOval() {
    const fh1 = Math.floor(Number(ovalFh1Input.value));
    const fv1 = Math.floor(Number(ovalFv1Input.value));
    const fh2 = Math.floor(Number(ovalFh2Input.value));
    const fv2 = Math.floor(Number(ovalFv2Input.value));
    if (!validNode(fh1, fv1) || !validNode(fh2, fv2)) {
      refreshUi('oval endpoints must be valid nodes');
      return;
    }
    ovals.push({ id: nextOvalId, fh1, fv1, fh2, fv2 });
    nextOvalId += 1;
    refreshUi('added oval');
  }

  function deleteOval() {
    const ovalId = selectedOvalId();
    if (!ovalId) {
      refreshUi('select an oval first');
      return;
    }
    ovals = ovals.filter(oval => oval.id !== ovalId);
    refreshUi('deleted oval');
  }

  function buildStepTargets(iterationIndex) {
    const step = getIteration(iterationIndex);
    return atoms.map(atom => {
      const fhKey = String(atom.fh);
      const fvKey = String(atom.fv);
      const fhActive = Object.prototype.hasOwnProperty.call(step.fh, fhKey);
      const fvActive = Object.prototype.hasOwnProperty.call(step.fv, fvKey);
      if (!fhActive || !fvActive) {
        return { ...atom };
      }
      const targetFh = atom.fh + Number(step.fh[fhKey]);
      const targetFv = atom.fv + Number(step.fv[fvKey]);
      if (targetFh < 0 || targetFh >= gridSize || targetFv < 0 || targetFv >= gridSize) {
        return { ...atom };
      }
      return { ...atom, fh: targetFh, fv: targetFv };
    });
  }

  function canApplyTargets(targetAtoms) {
    const occupancy = new Map();
    targetAtoms.forEach(atom => {
      const nodeKey = key(atom.fh, atom.fv);
      occupancy.set(nodeKey, (occupancy.get(nodeKey) || 0) + 1);
    });
    for (const count of occupancy.values()) {
      if (count > 2) {
        return false;
      }
    }
    return true;
  }

  function buildEffect(iterationIndex) {
    const step = getIteration(iterationIndex);
    const hadamardUids = new Set(step.hadamardUids.filter(uid => atoms.some(atom => atom.uid === uid)));
    const rydberg = Boolean(step.rydberg);
    return {
      hadamardUids,
      rydberg,
      rydbergUids: rydberg ? activeRydbergAtomUids() : new Set(),
      ovalIds: rydberg ? activeRydbergOvalIds() : new Set(),
    };
  }

  function addIteration() {
    if (animationState) {
      updateStatus('wait until animation finishes');
      return;
    }
    const newIndex = program.length;
    program.push(emptyIteration());
    requestedIterationIndex = newIndex;
    lastProgramConfigured = false;
    refreshUi(`added Iteration ${newIndex + 1}`);
  }

  function undoLastStep() {
    if (animationState) {
      updateStatus('wait until animation finishes');
      return;
    }
    if (stepHistory.length === 0) {
      refreshUi('nothing to undo');
      return;
    }
    const snapshot = stepHistory.pop();
    atoms = cloneAtoms(snapshot.atoms);
    activeEffect = null;
    refreshUi(`undid iteration ${snapshot.iterationIndex + 1}`);
  }

  function undoAllSteps() {
    if (animationState) {
      updateStatus('wait until animation finishes');
      return;
    }
    if (stepHistory.length === 0) {
      refreshUi('nothing to undo');
      return;
    }
    const snapshot = stepHistory[0];
    atoms = cloneAtoms(snapshot.atoms);
    stepHistory = [];
    activeEffect = null;
    refreshUi('returned to state before the first recorded step');
  }

  function startSteps(stepQueue, markConfigured) {
    if (animationState) {
      updateStatus('animation is already running');
      return;
    }
    if (atoms.length === 0) {
      refreshUi('add atoms first');
      return;
    }
    animationState = { stepQueue: stepQueue.slice(), phase: 'setup', duration: 1300 };
    if (markConfigured) {
      lastProgramConfigured = true;
    }
    prepareNextAnimationStep();
  }

  function stepSelectedIteration() {
    startSteps([activeIterationIndex()], true);
  }

  function runProgram() {
    startSteps(Array.from({ length: iterationCount() }, (_, index) => index), true);
  }

  function repeatProgram() {
    if (!lastProgramConfigured) {
      refreshUi('run a step or full program first');
      return;
    }
    startSteps(Array.from({ length: iterationCount() }, (_, index) => index), false);
  }

  function prepareNextAnimationStep() {
    if (!animationState) {
      return;
    }
    if (animationState.stepQueue.length === 0) {
      animationState = null;
      activeEffect = null;
      refreshUi('program completed');
      return;
    }
    const iterationIndex = animationState.stepQueue.shift();
    iterationSelect.value = String(iterationIndex);
    const step = getIteration(iterationIndex);
    const targetAtoms = buildStepTargets(iterationIndex);
    if (!canApplyTargets(targetAtoms)) {
      animationState = null;
      activeEffect = null;
      refreshUi(`run cancelled at iteration ${iterationIndex + 1}: more than two atoms would end up on one node`);
      return;
    }

    const startOccupancy = getNodeOccupancy(atoms);
    const endOccupancy = getNodeOccupancy(targetAtoms);
    const frames = sortAtoms(atoms).map(atom => {
      const start = atomPixel(atom, startOccupancy);
      const targetAtom = targetAtoms.find(candidate => candidate.uid === atom.uid);
      const end = atomPixel(targetAtom, endOccupancy);
      return { atom, start, end, x: start.x, y: start.y };
    });

    stepHistory.push({ iterationIndex, atoms: cloneAtoms(atoms) });
    activeEffect = buildEffect(iterationIndex);
    animationState = {
      ...animationState,
      phase: 'running',
      iterationIndex,
      step,
      sourceAtoms: atoms.map(atom => ({ ...atom })),
      targetAtoms,
      startOccupancy,
      frames,
      framePositions: new Map(),
      progress: 0,
      startedAt: performance.now(),
    };
    refreshIterationControls();
    updateStatus(`running iteration ${iterationIndex + 1} of ${iterationCount()}`);
    animate();
  }

  function animate() {
    if (!animationState || animationState.phase !== 'running') {
      return;
    }
    const now = performance.now();
    const progress = Math.min(1, (now - animationState.startedAt) / animationState.duration);
    const eased = 0.5 - 0.5 * Math.cos(Math.PI * progress);
    animationState.progress = eased;
    animationState.framePositions = new Map();
    animationState.frames.forEach(frame => {
      const x = frame.start.x + (frame.end.x - frame.start.x) * eased;
      const y = frame.start.y + (frame.end.y - frame.start.y) * eased;
      animationState.framePositions.set(frame.atom.uid, { x, y });
    });
    drawScene();
    if (progress < 1) {
      requestAnimationFrame(animate);
      return;
    }
    atoms = animationState.targetAtoms;
    activeEffect = null;
    animationState.phase = 'setup';
    prepareNextAnimationStep();
  }

  function buildConfig() {
    return {
      version: 2,
      gridSize,
      iterationCount: iterationCount(),
      coordinates: 'fh-fv-rotated',
      nextAtomId,
      nextAtomUid,
      nextOvalId,
      atoms: atoms.map(atom => ({ uid: atom.uid, type: atom.type, id: atom.id, fh: atom.fh, fv: atom.fv })),
      program: program.map(step => ({
        fh: { ...step.fh },
        fv: { ...step.fv },
        hadamardUids: step.hadamardUids.slice(),
        rydberg: Boolean(step.rydberg),
      })),
      ovals: ovals.map(oval => ({ ...oval })),
    };
  }

  function saveConfig() {
    const config = buildConfig();
    const blob = new Blob([JSON.stringify(config, null, 2)], { type: 'application/json' });
    const url = URL.createObjectURL(blob);
    const link = document.createElement('a');
    link.href = url;
    link.download = 'atom_grid_config.json';
    document.body.appendChild(link);
    link.click();
    link.remove();
    URL.revokeObjectURL(url);
    updateStatus('configuration saved to JSON');
  }

  function normalizeMovementMap(value) {
    const result = {};
    if (!value || typeof value !== 'object') {
      return result;
    }
    Object.keys(value).forEach(index => {
      const lineIndex = Number(index);
      if (Number.isInteger(lineIndex) && lineIndex >= 0 && lineIndex < gridSize) {
        result[String(lineIndex)] = Number(value[index]) || 0;
      }
    });
    return result;
  }

  function loadConfigObject(config) {
    if (!config || typeof config !== 'object') {
      throw new Error('Invalid JSON structure');
    }
    if (!Number.isInteger(config.gridSize) || config.gridSize < 2 || config.gridSize > 30) {
      throw new Error('gridSize must be an integer from 2 to 30');
    }
    gridSize = config.gridSize;
    sizeInput.value = gridSize;

    const rawAtoms = Array.isArray(config.atoms) ? config.atoms : [];
    const occupancy = new Map();
    atoms = rawAtoms.map(atom => {
      const fh = Number.isInteger(atom.fh) ? atom.fh : atom.row;
      const fv = Number.isInteger(atom.fv) ? atom.fv : atom.col;
      if (!Number.isInteger(atom.uid) || !Number.isInteger(atom.id)) {
        throw new Error('Each atom must have integer uid and id');
      }
      if (!['data', 'ancilla'].includes(atom.type)) {
        throw new Error('Atom type must be data or ancilla');
      }
      if (!validNode(fh, fv)) {
        throw new Error('Atom coordinates are outside the grid');
      }
      const nodeKey = key(fh, fv);
      occupancy.set(nodeKey, (occupancy.get(nodeKey) || 0) + 1);
      if (occupancy.get(nodeKey) > 2) {
        throw new Error('A node cannot contain more than two atoms');
      }
      return { uid: atom.uid, type: atom.type, id: atom.id, fh, fv };
    });

    const loadedProgramLength = Array.isArray(config.program)
      ? Math.max(1, config.program.length)
      : (Number.isInteger(config.iterationCount) && config.iterationCount > 0 ? config.iterationCount : INITIAL_ITERATION_COUNT);
    program = Array.from({ length: loadedProgramLength }, () => emptyIteration());
    if (Array.isArray(config.program)) {
      for (let i = 0; i < Math.min(program.length, config.program.length); i += 1) {
        const rawStep = config.program[i] || {};
        program[i] = {
          fh: normalizeMovementMap(rawStep.fh),
          fv: normalizeMovementMap(rawStep.fv),
          hadamardUids: Array.isArray(rawStep.hadamardUids) ? rawStep.hadamardUids.filter(uid => atoms.some(atom => atom.uid === uid)) : [],
          rydberg: Boolean(rawStep.rydberg),
        };
      }
    }

    ovals = Array.isArray(config.ovals) ? config.ovals.map(oval => ({
      id: Number.isInteger(oval.id) ? oval.id : nextOvalId,
      fh1: Number(oval.fh1),
      fv1: Number(oval.fv1),
      fh2: Number(oval.fh2),
      fv2: Number(oval.fv2),
    })).filter(oval => validNode(oval.fh1, oval.fv1) && validNode(oval.fh2, oval.fv2)) : [];

    nextAtomId = {
      data: Number.isInteger(config.nextAtomId?.data) ? config.nextAtomId.data : 1,
      ancilla: Number.isInteger(config.nextAtomId?.ancilla) ? config.nextAtomId.ancilla : 1,
    };
    nextAtomUid = Number.isInteger(config.nextAtomUid) ? config.nextAtomUid : (atoms.reduce((maxUid, atom) => Math.max(maxUid, atom.uid), 0) + 1);
    nextOvalId = Number.isInteger(config.nextOvalId) ? config.nextOvalId : (ovals.reduce((maxId, oval) => Math.max(maxId, oval.id), 0) + 1);
    renumberAtoms();
    selectedNode = null;
    clearUndoHistory();
    lastProgramConfigured = false;
    refreshUi('configuration loaded from JSON');
  }

  function handleLoadConfig(event) {
    if (animationState) {
      updateStatus('wait until animation finishes');
      event.target.value = '';
      return;
    }
    const file = event.target.files && event.target.files[0];
    if (!file) {
      return;
    }
    const reader = new FileReader();
    reader.onload = loadEvent => {
      try {
        loadConfigObject(JSON.parse(loadEvent.target.result));
      } catch (error) {
        refreshUi(`load failed: ${error.message}`);
      } finally {
        event.target.value = '';
      }
    };
    reader.onerror = () => {
      refreshUi('load failed: could not read file');
      event.target.value = '';
    };
    reader.readAsText(file);
  }

  canvas.addEventListener('click', event => {
    selectedNode = nearestNode(event);
    addAtomAt(selectedNode.fh, selectedNode.fv);
  });
  removeButton.addEventListener('click', removeSelectedNode);
  clearButton.addEventListener('click', clearGrid);
  stepButton.addEventListener('click', stepSelectedIteration);
  runButton.addEventListener('click', runProgram);
  repeatButton.addEventListener('click', repeatProgram);
  undoStepButton.addEventListener('click', undoLastStep);
  undoAllButton.addEventListener('click', undoAllSteps);
  saveButton.addEventListener('click', saveConfig);
  loadInput.addEventListener('change', handleLoadConfig);
  sizeInput.addEventListener('change', resizeGrid);
  iterationSelect.addEventListener('change', () => refreshUi(`switched to ${activeIterationLabel()}`));
  addIterationButton.addEventListener('click', addIteration);
  atomsSelect.addEventListener('change', drawScene);
  linesSelect.addEventListener('change', () => {
    const selected = selectedLine();
    if (selected) {
      axisInput.value = selected.axis;
      lineIndexInput.value = selected.index;
      lineDeltaInput.value = getIteration()[selected.axis][selected.index];
    }
  });
  setLineButton.addEventListener('click', setMovementLine);
  removeLineButton.addEventListener('click', removeMovementLine);
  clearLinesButton.addEventListener('click', clearMovementLines);
  setHButton.addEventListener('click', setHadamard);
  clearHButton.addEventListener('click', clearHadamard);
  rydbergInput.addEventListener('change', setRydbergFromInput);
  addOvalButton.addEventListener('click', addOval);
  deleteOvalButton.addEventListener('click', deleteOval);
  windowSizeInput.addEventListener('input', handleVisualControlChange);
  atomScaleInput.addEventListener('input', handleVisualControlChange);
  ovalWidthScaleInput.addEventListener('input', handleVisualControlChange);
  ovalLengthScaleInput.addEventListener('input', handleVisualControlChange);

  updateVisualSettings();
  refreshUi();
})();
</script>
"""

display(HTML(html.replace("__ID__", widget_id)))
